# 03 · Embed — 03 dimensions (the mismatch, demonstrated)

Several embedding dimensions are supported here:

| Model | Dimension |
|---|---|
| `text-embedding-3-large` (OpenAI) | 3072 |
| hash-v1 (offline default) | 384 |
| `BAAI/bge-large-en-v1.5` | 1024 |
| `nomic-embed-text` (Ollama) | 768 |

Mixing them does not raise an error at the point you'd expect. This notebook builds a tiny in-memory index at one dimension, then queries it with a vector built at a different dimension, and shows what actually happens — twice, two different ways, because the naive way and the "defensive" way fail differently, and the defensive way is the dangerous one.

No key and no network needed — everything here runs on the offline hash-embed path, just reconfigured to two different dimensions to stand in for two different models.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `hash_embed` | Deterministic offline embedding at a configurable dimension (numpy). | `hash_embed(text, dim=1024)` |
| `write_index_manifest` | Writes the JSON record of what model/dimension/tokenizer built an index. | `write_index_manifest(path, index_name=..., dimension=384, ...)` |
| `cosine_search` | Ranks index vectors against a query vector by cosine similarity. | `cosine_search(query_vec, index_vectors, texts, top_k=3)` |
| `coerce_to_index_dim` | Truncates or zero-pads a vector to force it to a target dimension — the "defensive" failure mode being demonstrated. | `coerce_to_index_dim(vec, 384)` |
| `assert_dimension_matches` | Raises if a query vector's dimension doesn't match the manifest's recorded dimension. | `assert_dimension_matches(manifest, query_vec)` |


## Step 1 — bootstrap the repo path and confirm the environment

Jupyter starts this kernel with the notebook's own directory as `cwd`, so `nbio` has to be located and put on `sys.path` before anything else can import it.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — define `hash_embed`, reproduced standalone

Same offline embedding as `01-offline-embeddings.ipynb`, reproduced here so this notebook runs standalone, expressed with numpy so the index can be one array instead of a list of lists.

In [ ]:
import hashlib

import numpy as np


def hash_embed(text: str, dim: int) -> np.ndarray:
    """Deterministic offline embedding at a configurable dimension."""
    vec = np.zeros(dim, dtype=np.float64)
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = np.linalg.norm(vec) or 1.0
    return vec / norm

## Step 3 — build a tiny index at 384 dimensions

Same five sample chunks as the other two notebooks in this stage.

In [ ]:
sample_chunks = [
    "The mitochondria is the powerhouse of the cell, converting nutrients into ATP through oxidative phosphorylation.",
    "Retrieval-augmented generation grounds a language model's answer in retrieved passages rather than parametric memory alone.",
    "A vector embedding maps a passage of text onto a point in a high-dimensional space so that semantic similarity becomes geometric distance.",
    "Photosynthesis converts light energy into chemical energy stored in glucose, releasing oxygen as a byproduct.",
    "An index manifest records the model, dimension and tokenizer used to build a vector store, so a mismatch is caught before it becomes a silent empty result.",
]

INDEX_DIM = 384  # a common default dimension for this scale of embedding
index_vectors = np.stack([hash_embed(c, INDEX_DIM) for c in sample_chunks])
print(f"index built: {index_vectors.shape[0]} vectors x {index_vectors.shape[1]} dim")

## Step 4 — define `write_index_manifest`

If this had been recorded the first time this index was queried with the wrong model, the check later in this notebook would have caught it before ever touching the index. `write_index_manifest()` is the fix: record what an index was built with, once, so a dimension mismatch becomes a one-line check instead of a debugging session.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path


def write_index_manifest(
    path: Path,
    *,
    index_name: str,
    model: str,
    dimension: int,
    tokenizer: str,
    max_tokens: int,
    overlap: int,
) -> dict:
    """The one JSON file that keeps 'index built at dimension X, queried
    with a model at dimension Y' from looking like an empty corpus instead
    of a config mismatch.
    """
    manifest = {
        "index_name": index_name,
        "model": model,
        "dimension": dimension,
        "tokenizer": tokenizer,
        "max_tokens": max_tokens,
        "overlap": overlap,
        "built_at": datetime.now(timezone.utc).isoformat(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2))
    return manifest

## Step 5 — write this index's manifest and read it back

In [ ]:
manifest_path = repo_root / "runs" / "demo-embed-dimensions" / "index_manifest.json"
manifest = write_index_manifest(
    manifest_path,
    index_name="demo-hash-384",
    model="hash-v1",
    dimension=INDEX_DIM,
    tokenizer="whitespace-split",
    max_tokens=None,
    overlap=0,
)
print(f"wrote {manifest_path}")
nbio.show_json(manifest)

## Step 6 — define `cosine_search`

A search function that looks like what a small local retrieval store actually does: cosine similarity (a plain dot product, since every vector here is already L2-normalized), ranked, with an optional relevance floor.

In [ ]:
def cosine_search(query_vec, index_vectors, texts, top_k=3, min_score=0.0):
    scores = index_vectors @ query_vec
    order = np.argsort(-scores)[:top_k]
    return [(texts[i], float(scores[i])) for i in order if scores[i] >= min_score]

## Step 7 — baseline: query at the matching dimension

Hash-embed vectors aren't semantically meaningful (see `01-offline-embeddings.ipynb`), so don't read anything into *which* chunk ranks first — the only thing this cell proves is that a matching-dimension query returns a normal, non-empty, scored result list. That's the contrast the mismatch below breaks.

In [ ]:
query = "What converts nutrients into ATP inside a cell?"
query_vec = hash_embed(query, INDEX_DIM)

print(f"query at matching dimension ({INDEX_DIM}):")
for text, score in cosine_search(query_vec, index_vectors, sample_chunks):
    print(f"  {score:+.4f}  {text[:60]!r}")

## Step 8 — Attempt A: the naive mismatch

Simulate switching to `BAAI/bge-large-en-v1.5`-shaped queries (1024 dim) against this same 384-dim index, without ever rebuilding it — the scenario this whole notebook is about: a store built at one dimension, queried with a model at another.

This is the naive way: compute the dot product directly, the way code does before anyone has thought about dimension safety at all.

In [ ]:
MISMATCHED_DIM = 1024  # simulating BAAI/bge-large-en-v1.5
mismatched_query_vec = hash_embed(query, MISMATCHED_DIM)

try:
    cosine_search(mismatched_query_vec, index_vectors, sample_chunks)
    print("no error raised")
except ValueError as exc:
    print(f"FAILED LOUD: {type(exc).__name__}: {exc}")

That's the loud failure — numpy refuses to multiply mismatched shapes, and the error names the exact dimensions involved. If every retrieval path looked like this, the mismatch this notebook is about wouldn't be a real problem.

## Step 9 — define `coerce_to_index_dim`, the "defensive" way

Some retrieval code pads or truncates a vector to fit, specifically so it never crashes on a shape mismatch. That instinct is exactly backwards: it trades a loud, immediate, informative error for a search that runs to completion and returns a plausible-looking, wrong answer.

In [ ]:
def coerce_to_index_dim(vec: np.ndarray, target_dim: int) -> np.ndarray:
    """What some retrieval code does to avoid ever crashing on a shape
    mismatch: truncate or zero-pad to fit. It never raises -- which is
    exactly the problem.
    """
    if len(vec) == target_dim:
        return vec
    if len(vec) > target_dim:
        return vec[:target_dim]
    out = np.zeros(target_dim)
    out[: len(vec)] = vec
    return out

## Step 10 — Attempt B: query with the coerced vector and look at the result

In [ ]:
MIN_SCORE = 0.25  # a relevance floor for filtering weak matches

coerced_query_vec = coerce_to_index_dim(mismatched_query_vec, INDEX_DIM)
results = cosine_search(coerced_query_vec, index_vectors, sample_chunks, min_score=MIN_SCORE)

print(f"query at coerced dimension (1024 -> {INDEX_DIM}), min_score={MIN_SCORE}: {len(results)} result(s)")
for text, score in results:
    print(f"  {score:+.4f}  {text[:60]!r}")
if not results:
    print("SILENT EMPTY RESULT -- no exception, no vectors returned. Indistinguishable from a corpus that genuinely has no relevant chunk.")

Run against this notebook's actual sample data, attempt B returned **zero results** — not an error, not a low-confidence match, nothing. A caller sees exactly what they'd see if the corpus simply didn't contain the answer. That is the failure mode this notebook is about: not a crash you'd notice and fix, but a silent, confident-looking empty result you have to go hunting for.

## The fix is the manifest, checked first

The index manifest written above isn't decoration — read back and compared *before* a query ever touches the index, it turns this exact bug into a one-line, immediate, loud error instead of a silent empty result.

## Step 11 — define `assert_dimension_matches`

In [ ]:
def assert_dimension_matches(manifest: dict, query_vec: np.ndarray) -> None:
    if manifest["dimension"] != len(query_vec):
        raise ValueError(
            f"index manifest says dimension={manifest['dimension']}, "
            f"query vector is dimension={len(query_vec)} -- stop, do not query"
        )

## Step 12 — run the guard against the mismatched query

In [ ]:
try:
    assert_dimension_matches(manifest, mismatched_query_vec)
except ValueError as exc:
    print(f"caught before ever touching the index: {exc}")

One dictionary lookup, checked before the search runs, is the difference between the silent empty result above and a loud, immediate, correct-cause error. Step 13 wires that check into the search path itself, rather than leaving it demonstrated but disconnected. See `README.md` for the full dimension table and `write_index_manifest()`'s contract.

## Step 13 — wire the guard INTO the search path, not just beside it

`cosine_search` (Step 6) and `assert_dimension_matches` (Step 11) have sat in this same notebook the whole time, unconnected — the guard was demonstrated, not used. `guarded_cosine_search` is the three-line fix: check the dimension first, then delegate to the real search. Run against the exact Attempt B scenario that silently returned zero results earlier, it now raises immediately instead.

In [ ]:
def guarded_cosine_search(manifest, query_vec, index_vectors, texts, top_k=3, min_score=0.0):
    assert_dimension_matches(manifest, query_vec)
    return cosine_search(query_vec, index_vectors, texts, top_k=top_k, min_score=min_score)


# Same coerced, wrong-dimension-in-spirit query that returned zero results
# silently under Attempt B. It never reaches cosine_search this time.
try:
    guarded_cosine_search(manifest, mismatched_query_vec, index_vectors, sample_chunks)
    print("FAILED: should have raised before searching")
except ValueError as exc:
    print(f"caught before the search ever ran: {exc}")

# The guard must not block a real, matching-dimension query.
clean_results = guarded_cosine_search(manifest, query_vec, index_vectors, sample_chunks)
assert len(clean_results) > 0, "a correctly-dimensioned query must still search normally"
print()
print(f"a matching-dimension query still returns {len(clean_results)} real result(s)")